# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and processing the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library.

### Dataset Source

The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets and their fields (by `@id`).

In [ ]:
# List all record sets and their field/column @ids
record_sets = []
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        print(f"RecordSet name: {rs.name}\n  @id: {rs.id}\n  Description: {rs.description}\n  Fields/columns:")
        record_sets.append(rs.id)
        # The fields/columns live in rs.fields
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}, type: {getattr(field, 'data_type', None)})")
else:
    print("No record sets found in the metadata. Trying alternative location...")
    # Try top-level attribute (older mlcroissant versions)
    for rs in dataset.record_sets:
        print(f"RecordSet name: {rs.name}\n  @id: {rs.id}\n  Description: {rs.description}\n  Fields/columns:")
        record_sets.append(rs.id)
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}, type: {getattr(field, 'data_type', None)})")
if not record_sets:
    raise ValueError("Could not find any record sets in this dataset.")

## 3. Data Extraction

Load data from each record set into a pandas DataFrame for analysis. All table and field references are by their `@id`, as above.

In [ ]:
# For this dataset, the main record set(s) is(are) typically named and specified in section above. We'll extract them all.
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records from RecordSet {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))  # Returns list of dicts
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in {record_set_id}: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found in {record_set_id}!")

# For demo, pick the first record set as main analysis target
main_record_set = record_sets[0]
main_df = dataframes[main_record_set]
print(f"Selected record set for EDA: {main_record_set}")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing: filter records on numeric fields, normalize, group, etc. All fields referenced by their `@id`.

In [ ]:
# Inspect column names to select numerical and categorical fields for analysis (by @id)
print("Available columns:", main_df.columns.tolist())
# Assume protein abundance is a numeric field. (Replace with actual field id as needed)
# For demonstration, search for likely numeric columns:
numeric_candidates = [col for col in main_df.columns if main_df[col].dtype.kind in 'fi']
if not numeric_candidates:
    # Try parsing all columns as float where possible
    for col in main_df.columns:
        try:
            main_df[col] = pd.to_numeric(main_df[col], errors='raise')
            numeric_candidates.append(col)
        except Exception:
            continue
print("Numeric columns detected:", numeric_candidates)

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]  # Use as demo
else:
    raise ValueError('No numeric field found to use for analysis!')
threshold = main_df[numeric_field_id].quantile(0.75)  # Use 75th percentile as example threshold

# Filtering records where numeric_field_id > threshold
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {{numeric_field_id}} > {{threshold}}:")
print(filtered_df.head())

# Normalize this numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to group by a protein description/categorical field
group_candidates = [col for col in main_df.columns if main_df[col].dtype == object and col != numeric_field_id]
if group_candidates:
    group_field_id = group_candidates[0]
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
    print(f"Grouped average of {numeric_field_id} by {group_field_id}:")
    print(grouped.head())
else:
    print("No categorical group field found.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of main numeric field
plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# If there is a categorical group field, plot a boxplot
if group_candidates:
    plt.figure(figsize=(12,5))
    sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion

In this notebook, you learned how to load and explore the FAIR^2 Croissant-based mass spectrometry protein dataset using the `mlcroissant` library. You accessed record sets and columns by their `@id`, extracted the data into DataFrames, performed basic filtering and normalization, and visualized the resulting features.

- The dataset provides high-quality tabular protein readouts, and is strongly structured by Croissant schema.
- You can adapt this EDA template to work with other Croissant datasets by changing the schema URL and referencing the appropriate `@id`s for record sets and fields.

For further analysis, refer to the dataset's [metadata description and methods](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and the [mlcroissant documentation](https://github.com/mlcommons/croissant-python).